# 运营仪表盘模板

这个 notebook 用于 business review / 运营复盘场景，复用现有的 notebook 框架中的聚合、时序、漏斗、分层和归因能力。

In [ ]:
from pathlib import Path
import pandas as pd

from notebook_framework import (
    NotebookProject,
    build_report_table,
    crosstab_summary,
    funnel_analysis,
    plot_metric_comparison,
    time_series_summary,
    attribution_summary,
    user_segmentation,
)

repo_root = Path.cwd().resolve().parent.parent
project = NotebookProject(repo_root=repo_root)
df = project.load_csv('notebook_env_demo.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

In [ ]:
summary = build_report_table(
    df,
    group_by='source',
    metrics={'visits': 'visits', 'orders': 'orders', 'revenue': 'revenue'},
)
summary

In [ ]:
source_trend = time_series_summary(df, date_col='date', value_col='visits', group_by='source', freq='D')
source_trend.head()

In [ ]:
cross = crosstab_summary(
    df,
    index_col='source',
    columns_col='region',
    values_col='revenue',
    aggfunc='sum',
)
cross

In [ ]:
funnel_df = df[['user_id', 'source', 'channel']].drop_duplicates().copy()
funnel_df['stage'] = 'view'
funnel = funnel_analysis(
    pd.DataFrame({'user_id': df['user_id'], 'stage': 'view'}),
    user_id_col='user_id',
    stage_col='stage',
)
funnel

In [ ]:
segments = user_segmentation(
    df.groupby('user_id', as_index=False).agg(revenue=('revenue', 'sum'))
    ,
    user_id_col='user_id',
    value_col='revenue',
)
segments.head()

In [ ]:
attribution = attribution_summary(
    df,
    user_id_col='user_id',
    channel_col='channel',
    value_col='revenue',
    conversion_col='orders',
    method='last_touch',
)
attribution

In [ ]:
fig, ax = plot_metric_comparison(
    summary,
    x_col='source',
    metrics=['visits', 'orders'],
    title='渠道流量与订单对比',
)
fig